# Feature Engineering Book

This notebook builds a single weekly, model-ready dataset from the project sources.

General Scope implemented here:
- Merge target prices with exogenous drivers (API, fuel, freight SPPI)
- Weekly alignment (`W-FRI`) using forward-fill for lower-frequency exogenous series
- Calendar features, lags, rolling statistics, and year-over-year deltas
- Structural-break indicators (2021Q4-2023Q1 energy shock window)
- Time-based train/test split and export to disk

Some Team notes I reflected:
- ARIMA-classical baseline support via lagged target and exogenous features
- Shared feature table suitable for ARIMAX, VAR/VARMA, ML, and deep models

In [14]:
import warnings
warnings.filterwarnings('ignore')

from pathlib import Path
import re
import numpy as np
import pandas as pd

pd.set_option('display.max_columns', 200)
pd.set_option('display.width', 180)

In [ ]:
# Paths and config
ROOT = Path.cwd()
DATA = '../data'
OUT_CSV = Path(DATA) / 'merged_features_weekly.csv'
OUT_PARQUET = Path(DATA) / 'merged_features_weekly.parquet'

WEEKLY_FREQ = 'W-FRI'
LAGS = [1, 2, 3, 4, 8, 12]
ROLL_WINDOWS = [4, 8]
TEST_START = pd.Timestamp('2025-01-03')  # first Friday in 2025

TARGET_FILE = Path(DATA) / 'cleaned_fruit_veg.csv'
API_FILE = Path(DATA) / 'API_20260129.csv'
FUEL_FILES = [
    Path(DATA) / 'weekly_road_fuel_prices_2003_to_2017.csv',
    Path(DATA) / 'weekly_road_fuel_prices_2018_to_now.csv',
]
SPPI_FILE = Path(DATA) / 'series-210226.csv'

assert TARGET_FILE.exists(), f'Missing {TARGET_FILE}'
assert API_FILE.exists(), f'Missing {API_FILE}'
assert SPPI_FILE.exists(), f'Missing {SPPI_FILE}'
for f in FUEL_FILES:
    assert f.exists(), f'Missing {f}'

print('Using data directory:', DATA)


Using data directory: data


## 1) Load target series (weekly commodity prices)

In [16]:
target = pd.read_csv(TARGET_FILE)
target['date'] = pd.to_datetime(target['date'])
target = target.rename(columns={'display_name': 'commodity', 'price': 'target_price'})

target = target[['date', 'commodity', 'target_price']].copy()
target = target.sort_values(['commodity', 'date']).reset_index(drop=True)

print('Target shape:', target.shape)
print('Date range:', target['date'].min().date(), 'to', target['date'].max().date())
print('Commodity count:', target['commodity'].nunique())
target.head()

Target shape: (4308, 3)
Date range: 2015-01-09 to 2026-02-16
Commodity count: 9


,date,commodity,target_price
0,2015-01-09,Beetroot,0.341626
1,2015-01-16,Beetroot,0.335645
2,2015-01-23,Beetroot,0.340148
3,2015-01-30,Beetroot,0.342772
4,2015-02-06,Beetroot,0.347403


## 2) Load and shape exogenous inputs

In [17]:
# API monthly indices
api = pd.read_csv(API_FILE)
api['date'] = pd.to_datetime(api['date'])

api_wide = api.pivot_table(index='date', columns='category', values='index', aggfunc='first').sort_index()

api_keep = [
    'energy_and_lubricants',
    'fertilisers_and_soil_improvers',
    'seeds',
    'plant_protection_products',
    'fresh_fruit',
    'fresh_vegetables',
]
api_keep = [c for c in api_keep if c in api_wide.columns]
api_wide = api_wide[api_keep].copy()
api_wide.columns = [f'api_{c}' for c in api_wide.columns]

print('API shape:', api_wide.shape)
api_wide.tail(3)

API shape: (143, 5)


,api_energy_and_lubricants,api_fertilisers_and_soil_improvers,api_plant_protection_products,api_fresh_fruit,api_fresh_vegetables
date,,,,,
2025-09-01,142.394407,174.970593,106.349871,134.562043,123.758743
2025-10-01,144.883785,171.890973,105.052744,117.913962,120.510885
2025-11-01,145.755948,173.881839,107.110876,115.885691,117.481978


In [18]:
# Weekly road fuel prices (robust column mapping)
fuel_raw = pd.concat([pd.read_csv(f) for f in FUEL_FILES], ignore_index=True)

def _pick_col(cols, patterns):
    cols_l = {c.lower(): c for c in cols}
    for p in patterns:
        for lc, orig in cols_l.items():
            if re.search(p, lc):
                return orig
    return None

date_col = _pick_col(fuel_raw.columns, [r'^date$', r'date'])
petrol_col = _pick_col(fuel_raw.columns, [r'ulsp', r'petrol'])
diesel_col = _pick_col(fuel_raw.columns, [r'ulsd', r'diesel'])

fuel = fuel_raw[[date_col, petrol_col, diesel_col]].copy()
fuel.columns = ['date', 'fuel_petrol_price', 'fuel_diesel_price']
fuel['date'] = pd.to_datetime(fuel['date'], errors='coerce')
for c in ['fuel_petrol_price', 'fuel_diesel_price']:
    fuel[c] = pd.to_numeric(fuel[c], errors='coerce')

fuel = fuel.dropna(subset=['date']).sort_values('date').drop_duplicates('date')
fuel = fuel.set_index('date')

print('Fuel shape:', fuel.shape)
fuel.tail(3)

Fuel shape: (467, 2)


,fuel_petrol_price,fuel_diesel_price
date,,
2026-05-01,134.90,144.19
2026-09-02,131.46,140.72
2026-12-01,133.43,142.63


In [19]:
# ONS SPPI CSV parser (extracts quarter values, then upsamples)
sppi_raw = pd.read_csv(SPPI_FILE, header=None)
quarter_rows = []

for i in range(len(sppi_raw)):
    row_vals = [str(v).strip() for v in sppi_raw.iloc[i].dropna().tolist()]
    if len(row_vals) < 2:
        continue
    label = row_vals[0]
    val = row_vals[-1]
    m = re.match(r'^(\d{4})\s*Q([1-4])$', label)
    if not m:
        continue
    year = int(m.group(1))
    q = int(m.group(2))
    month = q * 3
    date = pd.Timestamp(year=year, month=month, day=1) + pd.offsets.MonthEnd(0)
    value = pd.to_numeric(val, errors='coerce')
    if pd.notna(value):
        quarter_rows.append((date, value))

sppi = pd.DataFrame(quarter_rows, columns=['date', 'sppi_road_freight']).drop_duplicates('date')
sppi = sppi.sort_values('date').set_index('date')

print('SPPI shape:', sppi.shape)
sppi.tail(3)

SPPI shape: (119, 1)


,sppi_road_freight
date,
2025-03-31,127.2
2025-06-30,128.2
2025-09-30,128.7


## 3) Weekly alignment and merge

In [20]:
# Build shared weekly timeline anchored to the fruit/veg price dates (W-FRI)
start_date = target['date'].min()
end_date   = target['date'].max()
weekly_idx = pd.date_range(start=start_date, end=end_date, freq=WEEKLY_FREQ)

def resample_to_weekly(df, weekly_idx):
    """
    Correctly forward-fill a lower/different-frequency series onto a W-FRI index.

    Plain reindex(weekly_idx) only matches exact dates, so monthly (1st of month),
    quarterly (quarter-end), or Monday-weekly series produce almost entirely NaN
    before ffill has anything to propagate. The fix: union the source dates with
    the target weekly dates first, ffill across the combined index, then select
    only the weekly dates.
    """
    combined = df.index.union(weekly_idx)
    return df.reindex(combined).ffill().reindex(weekly_idx)

# Track pre-fill missingness: 1 where no new source observation fell in that week
api_w_raw  = api_wide.reindex(weekly_idx)   # NaN where no monthly date matched a Friday
fuel_w_raw = fuel.reindex(weekly_idx)        # NaN where no Monday matched a Friday
sppi_w_raw = sppi.reindex(weekly_idx)        # NaN where no quarter-end matched a Friday

exog_missing_flags = pd.concat([
    api_w_raw.isna().add_suffix('_was_missing'),
    fuel_w_raw.isna().add_suffix('_was_missing'),
    sppi_w_raw.isna().add_suffix('_was_missing'),
], axis=1).astype('int8')

# Resample each series to W-FRI using the union+ffill approach
api_w  = resample_to_weekly(api_wide, weekly_idx)
fuel_w = resample_to_weekly(fuel,     weekly_idx)
sppi_w = resample_to_weekly(sppi,     weekly_idx)

exog_weekly = pd.concat([api_w, fuel_w, sppi_w, exog_missing_flags], axis=1)
exog_weekly.index.name = 'date'
exog_weekly = exog_weekly.reset_index()

merged = target.merge(exog_weekly, on='date', how='left')
merged = merged.sort_values(['commodity', 'date']).reset_index(drop=True)

print('Merged shape:', merged.shape)
print('API non-null after fix:',  merged['api_energy_and_lubricants'].notna().sum(), '/', len(merged))
print('Fuel non-null after fix:', merged['fuel_petrol_price'].notna().sum(), '/', len(merged))
print('SPPI non-null after fix:', merged['sppi_road_freight'].notna().sum(), '/', len(merged))
merged.head(3)

Merged shape: (4308, 19)
API non-null after fix: 4073 / 4308
Fuel non-null after fix: 4073 / 4308
SPPI non-null after fix: 4073 / 4308


,date,commodity,target_price,api_energy_and_lubricants,api_fertilisers_and_soil_improvers,api_plant_protection_products,api_fresh_fruit,api_fresh_vegetables,fuel_petrol_price,fuel_diesel_price,sppi_road_freight,api_energy_and_lubricants_was_missing,api_fertilisers_and_soil_improvers_was_missing,api_plant_protection_products_was_missing,api_fresh_fruit_was_missing,api_fresh_vegetables_was_missing,fuel_petrol_price_was_missing,fuel_diesel_price_was_missing,sppi_road_freight_was_missing
0,2015-01-09,Beetroot,0.341626,91.688446,119.925622,68.339627,56.097455,72.977881,116.13,121.3,99.9,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0
1,2015-01-16,Beetroot,0.335645,91.688446,119.925622,68.339627,56.097455,72.977881,116.13,121.3,99.9,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0
2,2015-01-23,Beetroot,0.340148,91.688446,119.925622,68.339627,56.097455,72.977881,116.13,121.3,99.9,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0


## 4) Time features and intervention flags

In [21]:
iso = merged['date'].dt.isocalendar()
merged['year'] = merged['date'].dt.year
merged['month'] = merged['date'].dt.month
merged['quarter'] = merged['date'].dt.quarter
merged['weekofyear'] = iso.week.astype(int)

# Cyclical encoding for week-of-year
merged['week_sin'] = np.sin(2 * np.pi * merged['weekofyear'] / 52.0)
merged['week_cos'] = np.cos(2 * np.pi * merged['weekofyear'] / 52.0)

# Structural break window for energy shock
shock_start = pd.Timestamp('2021-10-01')
shock_end = pd.Timestamp('2023-03-31')
merged['shock_2021q4_2023q1'] = ((merged['date'] >= shock_start) & (merged['date'] <= shock_end)).astype('int8')
merged['post_shock'] = (merged['date'] > shock_end).astype('int8')

merged[['date', 'commodity', 'weekofyear', 'week_sin', 'week_cos', 'shock_2021q4_2023q1']].head(3)

,date,commodity,weekofyear,week_sin,week_cos,shock_2021q4_2023q1
0,2015-01-09,Beetroot,2,0.239316,0.970942,0
1,2015-01-16,Beetroot,3,0.354605,0.935016,0
2,2015-01-23,Beetroot,4,0.464723,0.885456,0


## 5) Lag, rolling, and YoY features

In [22]:
# Shared exogenous columns (numeric, non-target)
exclude_cols = {'date', 'commodity', 'target_price'}
num_cols = [c for c in merged.columns if c not in exclude_cols and pd.api.types.is_numeric_dtype(merged[c])]

# Commodity-level target dynamics
g = merged.groupby('commodity', group_keys=False)
for lag in LAGS:
    merged[f'target_lag_{lag}'] = g['target_price'].shift(lag)

for w in ROLL_WINDOWS:
    merged[f'target_roll_mean_{w}'] = g['target_price'].shift(1).rolling(w, min_periods=1).mean().reset_index(level=0, drop=True)
    merged[f'target_roll_std_{w}'] = g['target_price'].shift(1).rolling(w, min_periods=2).std().reset_index(level=0, drop=True)

merged['target_yoy_diff_52'] = g['target_price'].diff(52)
merged['target_yoy_pct_52'] = g['target_price'].pct_change(52)

# Exogenous lags and rolling stats (same for all commodities, merged row-wise)
exog_numeric = [c for c in num_cols if not c.endswith('_was_missing')]
for c in exog_numeric:
    for lag in LAGS:
        merged[f'{c}_lag_{lag}'] = merged[c].shift(lag)
    for w in ROLL_WINDOWS:
        merged[f'{c}_roll_mean_{w}'] = merged[c].shift(1).rolling(w, min_periods=1).mean()
    merged[f'{c}_yoy_diff_52'] = merged[c].diff(52)

print('Feature engineering complete. Columns:', len(merged.columns))


Feature engineering complete. Columns: 183


## 6) Final modeling table and split

In [23]:
# Keep rows with observed target only, then drop early rows lacking lags
model_df = merged.dropna(subset=['target_price']).copy()

# Minimum lag requirement (12-week lag + 52-week YoY implies ~52 initial rows dropped per commodity)
required_cols = [
    'target_lag_1', 'target_lag_4', 'target_lag_12',
    'target_yoy_diff_52',
]
model_df = model_df.dropna(subset=required_cols).reset_index(drop=True)

# Train/test flag for downstream evaluation
model_df['split'] = np.where(model_df['date'] < TEST_START, 'train', 'test')

print('Final table shape:', model_df.shape)
print('Train rows:', (model_df['split'] == 'train').sum())
print('Test rows:', (model_df['split'] == 'test').sum())
print('Commodities:', model_df['commodity'].nunique())

model_df.head(3)

Final table shape: (3840, 184)
Train rows: 3605
Test rows: 235
Commodities: 9


,date,commodity,target_price,api_energy_and_lubricants,api_fertilisers_and_soil_improvers,api_plant_protection_products,api_fresh_fruit,api_fresh_vegetables,fuel_petrol_price,fuel_diesel_price,sppi_road_freight,api_energy_and_lubricants_was_missing,api_fertilisers_and_soil_improvers_was_missing,api_plant_protection_products_was_missing,api_fresh_fruit_was_missing,api_fresh_vegetables_was_missing,fuel_petrol_price_was_missing,fuel_diesel_price_was_missing,sppi_road_freight_was_missing,year,month,quarter,weekofyear,week_sin,week_cos,shock_2021q4_2023q1,post_shock,target_lag_1,target_lag_2,target_lag_3,target_lag_4,target_lag_8,target_lag_12,target_roll_mean_4,target_roll_std_4,target_roll_mean_8,target_roll_std_8,target_yoy_diff_52,target_yoy_pct_52,api_energy_and_lubricants_lag_1,api_energy_and_lubricants_lag_2,api_energy_and_lubricants_lag_3,api_energy_and_lubricants_lag_4,api_energy_and_lubricants_lag_8,api_energy_and_lubricants_lag_12,api_energy_and_lubricants_roll_mean_4,api_energy_and_lubricants_roll_mean_8,api_energy_and_lubricants_yoy_diff_52,api_fertilisers_and_soil_improvers_lag_1,api_fertilisers_and_soil_improvers_lag_2,api_fertilisers_and_soil_improvers_lag_3,api_fertilisers_and_soil_improvers_lag_4,api_fertilisers_and_soil_improvers_lag_8,api_fertilisers_and_soil_improvers_lag_12,api_fertilisers_and_soil_improvers_roll_mean_4,api_fertilisers_and_soil_improvers_roll_mean_8,api_fertilisers_and_soil_improvers_yoy_diff_52,api_plant_protection_products_lag_1,api_plant_protection_products_lag_2,api_plant_protection_products_lag_3,api_plant_protection_products_lag_4,api_plant_protection_products_lag_8,api_plant_protection_products_lag_12,api_plant_protection_products_roll_mean_4,api_plant_protection_products_roll_mean_8,api_plant_protection_products_yoy_diff_52,api_fresh_fruit_lag_1,api_fresh_fruit_lag_2,api_fresh_fruit_lag_3,api_fresh_fruit_lag_4,api_fresh_fruit_lag_8,api_fresh_fruit_lag_12,api_fresh_fruit_roll_mean_4,api_fresh_fruit_roll_mean_8,api_fresh_fruit_yoy_diff_52,api_fresh_vegetables_lag_1,api_fresh_vegetables_lag_2,api_fresh_vegetables_lag_3,api_fresh_vegetables_lag_4,api_fresh_vegetables_lag_8,api_fresh_vegetables_lag_12,api_fresh_vegetables_roll_mean_4,api_fresh_vegetables_roll_mean_8,api_fresh_vegetables_yoy_diff_52,fuel_petrol_price_lag_1,fuel_petrol_price_lag_2,fuel_petrol_price_lag_3,fuel_petrol_price_lag_4,fuel_petrol_price_lag_8,fuel_petrol_price_lag_12,fuel_petrol_price_roll_mean_4,fuel_petrol_price_roll_mean_8,fuel_petrol_price_yoy_diff_52,fuel_diesel_price_lag_1,fuel_diesel_price_lag_2,fuel_diesel_price_lag_3,fuel_diesel_price_lag_4,fuel_diesel_price_lag_8,fuel_diesel_price_lag_12,fuel_diesel_price_roll_mean_4,fuel_diesel_price_roll_mean_8,fuel_diesel_price_yoy_diff_52,sppi_road_freight_lag_1,sppi_road_freight_lag_2,sppi_road_freight_lag_3,sppi_road_freight_lag_4,sppi_road_freight_lag_8,sppi_road_freight_lag_12,sppi_road_freight_roll_mean_4,sppi_road_freight_roll_mean_8,sppi_road_freight_yoy_diff_52,year_lag_1,year_lag_2,year_lag_3,year_lag_4,year_lag_8,year_lag_12,year_roll_mean_4,year_roll_mean_8,year_yoy_diff_52,month_lag_1,month_lag_2,month_lag_3,month_lag_4,month_lag_8,month_lag_12,month_roll_mean_4,month_roll_mean_8,month_yoy_diff_52,quarter_lag_1,quarter_lag_2,quarter_lag_3,quarter_lag_4,quarter_lag_8,quarter_lag_12,quarter_roll_mean_4,quarter_roll_mean_8,quarter_yoy_diff_52,weekofyear_lag_1,weekofyear_lag_2,weekofyear_lag_3,weekofyear_lag_4,weekofyear_lag_8,weekofyear_lag_12,weekofyear_roll_mean_4,weekofyear_roll_mean_8,weekofyear_yoy_diff_52,week_sin_lag_1,week_sin_lag_2,week_sin_lag_3,week_sin_lag_4,week_sin_lag_8,week_sin_lag_12,week_sin_roll_mean_4,week_sin_roll_mean_8,week_sin_yoy_diff_52,week_cos_lag_1,week_cos_lag_2,week_cos_lag_3,week_cos_lag_4,week_cos_lag_8,week_cos_lag_12,week_cos_roll_mean_4,week_cos_roll_mean_8,week_cos_yoy_diff_52,shock_2021q4_2023q1_lag_1,shock_2021q4_2023q1_lag_2,shock_2021q4_2023q1_lag_3,shock_2021q4_2023q1_lag_4,shock_2021q4_2023q1_lag_8,shock_2021q4_2023q1_

In [24]:
# Save outputs
model_df.to_csv(OUT_CSV, index=False)

try:
    model_df.to_parquet(OUT_PARQUET, index=False)
    parquet_msg = f'Saved: {OUT_PARQUET}'
except Exception as e:
    parquet_msg = f'Parquet not saved ({type(e).__name__}: {e})'

print(f'Saved: {OUT_CSV}')
print(parquet_msg)


Saved: data\merged_features_weekly.csv
Saved: data\merged_features_weekly.parquet


## 7) Quick quality checks

In [25]:
summary = {
    'rows': len(model_df),
    'columns': len(model_df.columns),
    'date_min': model_df['date'].min(),
    'date_max': model_df['date'].max(),
    'commodities': model_df['commodity'].nunique(),
}
print(summary)

missing_top = model_df.isna().mean().sort_values(ascending=False).head(15)
missing_top

{'rows': 3840, 'columns': 184, 'date_min': Timestamp('2016-01-22 00:00:00'), 'date_max': Timestamp('2026-02-16 00:00:00'), 'commodities': 9}


api_energy_and_lubricants                         0.061198
api_fresh_fruit                                   0.061198
api_plant_protection_products                     0.061198
api_fertilisers_and_soil_improvers                0.061198
api_fresh_vegetables                              0.061198
sppi_road_freight                                 0.061198
fuel_diesel_price                                 0.061198
fuel_petrol_price                                 0.061198
fuel_petrol_price_was_missing                     0.061198
fuel_diesel_price_was_missing                     0.061198
sppi_road_freight_was_missing                     0.061198
api_energy_and_lubricants_was_missing             0.061198
api_fertilisers_and_soil_improvers_was_missing    0.061198
api_plant_protection_products_was_missing         0.061198
api_fresh_fruit_was_missing                       0.061198
dtype: float64